[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/milestone-assignment/assignment.ipynb)

# Module 3 milestone

Build the text-intelligence layer for a support inbox. See `README.md` for the full brief and the expected output of each part. Needs internet for the first model download.

## Setup

```
pip install transformers sentence-transformers
```

## The scenario and the fixed dataset

Run this cell as is. Every part below works on these fixed inputs. Do not change them.

In [6]:
messages = [
    "I was charged twice for my subscription this month and I want a refund.",
    "My card was declined at checkout but I was still billed for the upgrade.",
    "The app crashes every time I open the bank reconciliation screen.",
    "After the latest update my reports show the wrong totals, this is broken.",
    "How do I export my invoices to a CSV file?",
    "Where is the setting to add a second user to my account?",
    "Please cancel my subscription, I do not need the service any more.",
    "I want to close my account and stop all future payments.",
]

teams = {
    "Billing":      "A question or complaint about charges, payments, refunds, invoices, or subscription fees.",
    "Bug report":   "A report that the software is broken, crashing, showing errors, or giving wrong results.",
    "How-to":       "A question about how to use a feature or where to find a setting in the product.",
    "Cancellation": "A request to cancel the subscription or close the account.",
}

faqs = [
    "To export invoices, open Sales, then Invoices, select the rows, and choose Export to CSV.",
    "To add a user, open Settings, then Manage Users, click Add User, and set their role.",
    "To change your payment method, open Billing, then Payment Methods, and edit the card on file.",
    "To cancel your plan, open Billing, then Subscription, and choose Cancel Subscription.",
    "To reconcile an account, open Accounting, then Reconcile, and pick the account and statement date.",
    "If report totals look wrong, open Reports, then Settings, and refresh the data or clear the cache.",
]

for i, m in enumerate(messages, 1):
    print(i, m)

1 I was charged twice for my subscription this month and I want a refund.
2 My card was declined at checkout but I was still billed for the upgrade.
3 The app crashes every time I open the bank reconciliation screen.
4 After the latest update my reports show the wrong totals, this is broken.
5 How do I export my invoices to a CSV file?
6 Where is the setting to add a second user to my account?
7 Please cancel my subscription, I do not need the service any more.
8 I want to close my account and stop all future payments.


### Part 1. Tokenization audit (class 3.1, Text as data)

Load the `bert-base-uncased` tokenizer. For each message print the word count, the token count, and the token pieces. Then print the pieces for a few domain words and note which ones split.

In [7]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tok = AutoTokenizer.from_pretrained(model_name)

for msg in messages:
    print(f"\nMessage: {msg}")
    print(f"Word count: {len(msg.split())}")
    print(f"Token count: {len(tok.tokenize(msg))}")
    print(f"Tokens: {tok.tokenize(msg)}")

domain_words = ["reconciliation", "invoices", "csv"]

for w in domain_words:
    print(f"{w}: {tok.tokenize(w)}")

# Token count matters more than word count because
# LLMs process text as tokens, not complete words.
# Token count determines context-window usage and contributes
# to API costs and processing requirements.


Message: I was charged twice for my subscription this month and I want a refund.
Word count: 14
Token count: 16
Tokens: ['i', 'was', 'charged', 'twice', 'for', 'my', 'subscription', 'this', 'month', 'and', 'i', 'want', 'a', 'ref', '##und', '.']

Message: My card was declined at checkout but I was still billed for the upgrade.
Word count: 14
Token count: 16
Tokens: ['my', 'card', 'was', 'declined', 'at', 'check', '##out', 'but', 'i', 'was', 'still', 'billed', 'for', 'the', 'upgrade', '.']

Message: The app crashes every time I open the bank reconciliation screen.
Word count: 11
Token count: 12
Tokens: ['the', 'app', 'crashes', 'every', 'time', 'i', 'open', 'the', 'bank', 'reconciliation', 'screen', '.']

Message: After the latest update my reports show the wrong totals, this is broken.
Word count: 13
Token count: 15
Tokens: ['after', 'the', 'latest', 'update', 'my', 'reports', 'show', 'the', 'wrong', 'totals', ',', 'this', 'is', 'broken', '.']

Message: How do I export my invoices to a

### Part 2. Sentiment triage (class 3.4, Using pretrained transformers)

Score every message with the sentiment pipeline, flag confident negatives, and sort so the angriest come first.

In [8]:
from transformers import pipeline

# 1. Build a "sentiment-analysis" pipeline and score all messages.
# 2. Flag a message HIGH priority when it is NEGATIVE with score > 0.9.
# 3. Sort so the most negative print first.
# 4. In a comment, name the limitation this model shows on the how-to questions.
# your code here

classifier = pipeline("sentiment-analysis")

results = []

for msg in messages:
    prediction = classifier(msg)[0]

    label = prediction["label"]
    score = prediction["score"]

    priority = "HIGH" if label == "NEGATIVE" and score > 0.9 else "NORMAL"

    results.append({
        "message": msg,
        "label": label,
        "score": score,
        "priority": priority
    })

results.sort(
    key=lambda x: x["score"] if x["label"] == "NEGATIVE" else -1,
    reverse=True
)

for result in results:
    print(result)

# Limitation: This model performs sentiment classification,
# not question understanding. It may misinterpret how-to questions
# because it does not understand their actual intent.

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3808.04it/s]


{'message': 'After the latest update my reports show the wrong totals, this is broken.', 'label': 'NEGATIVE', 'score': 0.9997691512107849, 'priority': 'HIGH'}
{'message': 'I want to close my account and stop all future payments.', 'label': 'NEGATIVE', 'score': 0.9997658133506775, 'priority': 'HIGH'}
{'message': 'The app crashes every time I open the bank reconciliation screen.', 'label': 'NEGATIVE', 'score': 0.9996910095214844, 'priority': 'HIGH'}
{'message': 'Where is the setting to add a second user to my account?', 'label': 'NEGATIVE', 'score': 0.9995614886283875, 'priority': 'HIGH'}
{'message': 'How do I export my invoices to a CSV file?', 'label': 'NEGATIVE', 'score': 0.9992408752441406, 'priority': 'HIGH'}
{'message': 'Please cancel my subscription, I do not need the service any more.', 'label': 'NEGATIVE', 'score': 0.9990482926368713, 'priority': 'HIGH'}
{'message': 'I was charged twice for my subscription this month and I want a refund.', 'label': 'NEGATIVE', 'score': 0.9960587

### Part 3. Routing and FAQ retrieval with embeddings (classes 3.2, 3.3, 3.4)

First mean-pool a DistilBERT `last_hidden_state` to compare a paraphrase pair against an unrelated sentence. Then use a sentence-embedding model to route every message to the nearest team, and retrieve the best FAQ for the two how-to questions.

In [82]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from rich import print

# 1. Write embed_meanpool(text): run distilbert-base-uncased, mean-pool the
#    last_hidden_state using the attention mask, return one vector.
# 2. Show cosine(paraphrase pair) is higher than cosine(unrelated pair).
# your code here
model = AutoModel.from_pretrained("distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def embed_meanpool(text):
    inputs = tokenizer(text, return_tensors='pt', padding=True, truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)

    token_embedding = outputs.last_hidden_state

    attention_mask = inputs["attention_mask"]
    mask = attention_mask.unsqueeze(-1).float()

    summed = (token_embedding * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)

    return summed / counts

s1 = "The customer paid the invoice."
s2 = "The invoice was paid by the customer."
s3 = "Python is a programming language."

e1 = embed_meanpool(s1)
e2 = embed_meanpool(s2)
e3 = embed_meanpool(s3)

similarity1 = F.cosine_similarity(e1, e2)
similarity2 = F.cosine_similarity(e1, e3)

print("Paraphrase similarity:", similarity1.item())
print("Unrelated similarity:", similarity2.item())

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3257.41it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Paraphrase similarity: 0.9764100313186646

Unrelated similarity: 0.5111892819404602

In [85]:
from sentence_transformers import SentenceTransformer, util

# 3. Load "all-MiniLM-L6-v2". Encode the messages and the four team descriptions
#    (list(teams.values())) with normalize_embeddings=True.
# 4. Route each message to the team with the highest cosine similarity and print it.
# your code here
model = SentenceTransformer("all-MiniLM-L6-v2")

message_embeddings = model.encode(
    messages,
    normalize_embeddings=True,
    convert_to_tensor=True
)

team_names = list(teams.keys())

team_embeddings = model.encode(
    list(teams.values()),
    normalize_embeddings=True,
    convert_to_tensor=True
)

similarities = util.cos_sim(
    message_embeddings,
    team_embeddings
)

for i, msg in enumerate(messages):

    best_team_index = similarities[i].argmax().item()

    best_team = team_names[best_team_index]

    score = similarities[i][best_team_index].item()

    print(f"Message: {msg}")
    print(f"Assigned Team: {best_team}")
    print(f"Similarity: {score:.4f}")
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9850.27it/s]


Message: I was charged twice for my subscription this month and I want a refund.

Assigned Team: Billing

Similarity: 0.5601

--------------------------------------------------

Message: My card was declined at checkout but I was still billed for the upgrade.

... (23 outputs left out)

Message: I want to close my account and stop all future payments.

Assigned Team: Cancellation

Similarity: 0.6032

--------------------------------------------------

In [123]:
# 5. Encode the faqs. For messages 5 and 6 (the how-to questions), print the
#    single best-matching FAQ entry.
# your code here
quetions = messages[4:6]
encoded_faqs = model.encode(faqs, normalize_embeddings=True, convert_to_tensor=True)
encoded_msg = model.encode(quetions, normalize_embeddings=True, convert_to_tensor=True)

cos_sim = util.cos_sim(encoded_msg, encoded_faqs)

for i, sim in enumerate(cos_sim):
    best_match_faq = cos_sim[i].argmax().item()
    best_faq = faqs[best_match_faq]
    
    score = cos_sim[i][best_match_faq].item()

    print(f"Question: {quetions[i]}")
    print(f"best matching faq: {best_faq}")
    print(f"score: {score:.3f}")

Question: How do I export my invoices to a CSV file?

best matching faq: To export invoices, open Sales, then Invoices, select the rows, and choose Export to CSV.

score: 0.837

Question: Where is the setting to add a second user to my account?

best matching faq: To add a user, open Settings, then Manage Users, click Add User, and set their role.

score: 0.669

### Part 4. Draft a reply and control the decoding (class 3.5, From models to LLMs)

Use `distilgpt2` to draft a reply to message 5, three ways: greedy, high temperature, and top-p. Print the prompt token count against the context window.

In [146]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch

# 1. Load distilgpt2 (tokenizer + AutoModelForCausalLM).
# 2. Build a prompt from message 5 and print prompt tokens vs config.n_positions.
# 3. Generate 40 new tokens three times from the same prompt, calling set_seed(0)
#    before each: greedy (do_sample=False); temperature=1.3, top_k=0; top_p=0.9, top_k=0.
# 4. In a comment, say how the three drafts differ.
# your code here
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
model = AutoModelForCausalLM.from_pretrained("distilgpt2")

prompt = messages[6]

inputs = tokenizer(prompt, return_tensors="pt")

prompt_tokens = inputs["input_ids"].shape[1]
max_positions = model.config.n_positions

print("Prompt:", prompt)
print("Prompt tokens:", prompt_tokens)
print("Maximum context positions:", max_positions)


set_seed(0)

output_greedy = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False
)

print("\nGREEDY:")
print(tokenizer.decode(output_greedy[0], skip_special_tokens=True))


set_seed(0)

output_temperature = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    temperature=1.3,
    top_k=0
)

print("\nTEMPERATURE 1.3:")
print(tokenizer.decode(output_temperature[0], skip_special_tokens=True))


set_seed(0)

output_top_p = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=True,
    top_p=0.9,
    top_k=0
)

print("\nTOP-P 0.9:")
print(tokenizer.decode(output_top_p[0], skip_special_tokens=True))


# Observation:
# Greedy always selects the highest-probability token.
# Temperature 1.3 produces more randomness and diverse wording.
# Top-p samples from the smallest group of tokens covering 90%
# cumulative probability, balancing diversity and relevance.


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 10606.83it/s]


Prompt: Please cancel my subscription, I do not need the service any more.

Prompt tokens: 14

Maximum context positions: 1024

GREEDY:

Please cancel my subscription, I do not need the service any more.

TEMPERATURE 1.3:

Please cancel my subscription, I do not need the service any more. Please button seven toave away your Goldbacks at

TOP-P 0.9:

Please cancel my subscription, I do not need the service any more. Please read the Terms of Service and is 
requesting the full removal of my usage privileges, and if you would like to be notified of any issues, please 
contact Customer Service in a meaningful way.

### Write-up

Write four or five sentences here. How did tokenization shape the token counts and cost in Part 1? How does attention (each token pulling in context from the rest) explain the sentiment flips in Part 2 and the routing similarities in Part 3?

_Your answer:_

Tokenization breaks text into smaller units called tokens, so token count depends on the words, subwords, and punctuation rather than the number of words alone. More tokens generally mean higher processing costs because models process token sequences, and longer sequences require more computation. Attention allows each token to gather information from other relevant tokens, helping the model understand context rather than interpreting words independently. This explains sentiment flips because words like "not" can change the meaning of an entire sentence depending on their context. Similarly, in routing, attention helps identify relationships between words and capture semantic similarities, allowing sentences with different wording but similar meanings to produce similar representations.